# Solution B2 · Regression trees on house prices

**Type**: Implement with scikit-learn

## Objective

Compare a linear regression with a regression tree on the California prices, look at how the validation RMSE changes as `max_depth` and `min_samples_leaf` vary, choose both with a search, and check how much the geographic position is worth (a tree on latitude and
longitude alone).

In [ ]:
import sys, pathlib
_root = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "common").is_dir())
sys.path.insert(0, str(_root))

import warnings; warnings.filterwarnings("ignore")     # clean output: here the warnings are just noise
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from common import datasets as ds
from common.verify import check, close, done, verify

from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score, train_test_split
from sklearn.tree import DecisionTreeRegressor

X, y = ds.california()
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, random_state=0)
cv = KFold(5, shuffle=True, random_state=0)
error = lambda model, columns=None: mean_squared_error(y_te, model.fit(X_tr if columns is None else X_tr[columns], y_tr).predict(X_te if columns is None else X_te[columns]), squared=False)

## Commented solution

In [ ]:
rmse = {"linear": error(LinearRegression()), "base tree": error(DecisionTreeRegressor(random_state=0))}
print({k: round(v, 3) for k, v in rmse.items()})

In [ ]:
def cv_rmse(**kw):
    return -cross_val_score(DecisionTreeRegressor(random_state=0, **kw), X_tr, y_tr, cv=cv, scoring="neg_root_mean_squared_error").mean()


depth_curve = pd.Series({d: cv_rmse(max_depth=d) for d in [3, 5, 8, 12, 50]})
leaf_curve = pd.Series({l: cv_rmse(min_samples_leaf=l) for l in [1, 5, 20, 50, 100]})
best_depth, best_leaf = depth_curve.idxmin(), leaf_curve.idxmin()
print("cross-validation RMSE by max_depth:", depth_curve.round(3).to_dict())
print("cross-validation RMSE by min_samples_leaf:", leaf_curve.round(3).to_dict())

In [ ]:
search = GridSearchCV(DecisionTreeRegressor(random_state=0), {"max_depth": [6, 8, 10, 12, None], "min_samples_leaf": [5, 10, 20, 40]}, cv=cv,
                      scoring="neg_root_mean_squared_error").fit(X_tr, y_tr)
params = search.best_params_
tree_rmse = mean_squared_error(y_te, search.predict(X_te), squared=False)
n_leaves = search.best_estimator_.get_n_leaves()
print(f"best parameters: {params} | test RMSE {tree_rmse:.3f} (linear {rmse['linear']:.3f}) | leaves {n_leaves}")

In [ ]:
geo_rmse = error(DecisionTreeRegressor(min_samples_leaf=20, random_state=0), ["Latitude", "Longitude"])
geo_is_enough = bool(geo_rmse <= tree_rmse + 0.05)
print(f"tree on the coordinates only: RMSE {geo_rmse:.3f} | within 0.05 of the full tree: {geo_is_enough}")

## Check

In [ ]:
with verify():
    check(set(rmse) == {"linear", "base tree"} and abs(rmse["base tree"] - rmse["linear"]) < 0.03, "the unregularized tree (0.74) is on par with linear regression (0.735)", "error(model) with squared=False")
    check(list(depth_curve.index) == [3, 5, 8, 12, 50] and best_depth == 8 and depth_curve[8] < depth_curve[50] - 0.05, "the best depth is 8; with no limit (50) the RMSE is over 0.05 higher", "idxmin of the curve")
    check(list(leaf_curve.index) == [1, 5, 20, 50, 100] and best_leaf in (20, 50) and leaf_curve[1] > leaf_curve[best_leaf] + 0.1, "best min_samples_leaf 20 or 50; with 1 the RMSE is over 0.1 higher", "idxmin of the curve")
    check(params["min_samples_leaf"] in (10, 20, 40) and tree_rmse < 0.65 and tree_rmse < rmse["linear"] - 0.08, "the optimized tree beats the linear one by over 0.08 (RMSE below 0.65)", "GridSearchCV on max_depth and min_samples_leaf")
    check(n_leaves >= 200, "the best tree has hundreds of leaves: unlike the Titanic many rules are needed here", "get_n_leaves() of the final model")
    check(geo_rmse < 0.66 and geo_is_enough is True, "with the geographic position alone the RMSE is within 0.05 of the full tree", "DecisionTreeRegressor on the two coordinates only")
done()

## What you learned

- **An unregularized tree is worth as much as a straight line** (RMSE 0.74 against 0.735): it memorizes the 15,000 examples and generalizes like a linear model. With `min_samples_leaf=20` it drops to 0.62 (−16%): the minimum leaf size is the most effective regularization, because it ties the stability of the leaf means to the amount of data.
- **Hundreds of leaves** are needed (605): the phenomenon is complex (interactions between place, income and age) and a tree with 20 leaves would not be enough. On the Titanic about twenty rules were enough; here the readable rules are too many.
- **The geographic position alone is enough:** with latitude and longitude the RMSE is 0.61, as with all the features (0.62): the price depends mostly on where you are. Remember the KNN of module 5? With 0.53 it is even better: the step predictions of a tree are less suited to a smooth surface in space.
- The choices of `max_depth` and `min_samples_leaf` partly replace each other: with large leaves the depth is limited in practice. Search the two together, or fix one and search the other.

## Common pitfall

Choosing `max_depth` without looking at `min_samples_leaf`: with a large `min_samples_leaf` the depth is not needed, and with limited depth the leaves stay large even without the rule. The two parameters overlap and have to be evaluated together.

## Interview question

> *Click on a question to show its answer.*

<details><summary>How do you regularize a regression tree?</summary>

With the minimum leaf size (`min_samples_leaf`), which is the most direct control of the variance, with the maximum depth and with cost-complexity pruning (`ccp_alpha`), choosing in cross-validation. An unregularized tree memorizes the training set: here it was worth as much as a straight line,
with the right minimum leaf size it beat the line by over 15%.

</details>